# RAG 消融與來源評估

[完整講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e4-rag-ablation.md) · [在 Colab 執行](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/e4_rag_ablation.ipynb)

本練習使用自編小型資料，在 CPU 執行。用同一組題目比較三種條件，逐題找出檢索、答案與引用的差異。

如果 RAG 的回答變準，要判斷是檢索找對文件，還是生成模型本來就知道答案。**消融實驗（ablation）一次改動一項設計**。在這個自編例子中，固定四道題、回答規則和評分，改變提供給生成器的文件。

| 條件 | 生成器拿到什麼？ | 想理解的問題 |
|---|---|---|
| 無檢索 | 沒有提供文件 | 模型單獨作答的基準 |
| Top-2 檢索 | 檢索結果最前面的兩份 | 一般檢索和生成合作時的表現 |
| 人工提供正確文件（oracle） | 人工確認的證據文件 | 已知有證據時，生成端還會出哪些錯誤？ |

## 同一題可能答對、也可能缺少證據

每題先標一份正解文件，再分開記錄三件事。**檢索命中**看輸入給模型的文件裡是否有正解來源。**答案正確**由答案與標準答案比對。**答對且引用命中**要求答案正確、引用正解文件，而且該文件確實有提供給模型。這是教學版引用核對，正式評估還要檢查被引用段落能否支撐回答的具體敘述。

自編四題的結果如下，分母都是四題。延遲是預先填入的示例毫秒數。

| 條件 | 檢索命中 | 答案正確 | 答對且引用命中 | 延遲中位數 |
|---|---:|---:|---:|---:|
| 無檢索 | 0/4 | 2/4 | 0/4 | 32.5 ms |
| Top-2 | 3/4 | 3/4 | 2/4 | 100 ms |
| 人工文件 | 4/4 | 3/4 | 3/4 | 60 ms |

Top-2 和人工文件的答對率同為 3/4，但兩者的來源指標不同。人工已提供正確文件時仍有一題答錯，值得回看生成器怎麼使用證據；Top-2 中也有一題答對但沒取到正解文件。沿用第 09 課，把同一題配對比較，才能具體指出是哪種情況。

## 執行環境

Colab 通常已有 NumPy 和 Matplotlib。若要使用本專案規定範圍內的套件，可執行下一格。

In [ ]:
%pip -q install 'numpy>=1.26,<3' 'matplotlib>=3.8,<4'

## 程式驗證

先預測結果，再執行下方的手算核對和視覺化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gold_answers = ["30", "72", "blue", "June"]
gold_sources = ["d1", "d2", "d3", "d4"]
# Each row is (retrieved IDs, answer, cited ID, illustrative latency in ms).
trials = {
    "No retrieval": [
        ([], "30", None, 25), ([], "48", None, 30),
        ([], "blue", None, 35), ([], "May", None, 40),
    ],
    "Top-2": [
        (["d1", "x"], "30", "d1", 80), (["d2", "x"], "48", "d2", 90),
        (["x", "y"], "blue", "x", 110), (["x", "d4"], "June", "d4", 120),
    ],
    "Oracle": [
        (["d1"], "30", "d1", 50), (["d2"], "48", "d2", 55),
        (["d3"], "blue", "d3", 65), (["d4"], "June", "d4", 70),
    ],
}

def summarize(rows):
    hits, correct, cited, latencies = [], [], [], []
    for (retrieved, answer, reference, elapsed), expected, source in zip(rows, gold_answers, gold_sources):
        hit = source in retrieved
        right = answer == expected
        hits.append(hit)
        correct.append(right)
        cited.append(right and hit and reference == source)
        latencies.append(elapsed)
    return np.array([np.mean(hits), np.mean(correct), np.mean(cited), np.median(latencies)])

summary = np.stack([summarize(rows) for rows in trials.values()])
np.testing.assert_allclose(summary, [[0, .5, 0, 32.5], [.75, .75, .5, 100], [1, .75, .75, 60]])
for condition, (hit, acc, cited, latency) in zip(trials, summary):
    print(f"{condition}: hit={hit:.0%}, correct={acc:.0%}, correct with source={cited:.0%}, latency={latency:g} ms")
fig, ax = plt.subplots()
positions = np.arange(len(trials))
for i, name in enumerate(["Retrieved gold", "Correct answer", "Correct with gold citation"]):
    ax.bar(positions + (i - 1) * .24, summary[:, i], width=.22, label=name)
ax.set(xticks=positions, xticklabels=list(trials), ylabel="Fraction of questions", ylim=(0, 1.05), title="Separate retrieval, answer, and evidence")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2)
fig.tight_layout()
plt.show()

## 自我檢核與練習

把 Top-2 第三題補入正解文件並引用它，再重算三項指標。答案已正確，所以準確率仍為 3/4；檢索命中升為 4/4，答對且引用命中升為 3/4。再說明如何在真實實驗用重複量測記錄延遲。

## 來源

- [RAG 原論文](https://arxiv.org/abs/2005.11401)
- [RAG 綜述](https://arxiv.org/abs/2312.10997)